In [1]:
import pandas as pd
import numpy as np

In [42]:
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv')

In [43]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [44]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [45]:
numerical =['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
categorical =['lead_source', 'industry', 'employment_status', 'location']

In [46]:
df[numerical] = df[numerical].fillna(0)
df[categorical] = df[categorical].fillna('NA')
df

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NA,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1


In [47]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [48]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [49]:
df[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [50]:
from sklearn.model_selection import train_test_split

In [51]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [52]:
from sklearn.metrics import mutual_info_score

In [53]:
for cat in categorical:
    score = mutual_info_score(df_train[cat], df_train.converted)
    print(f'category= {cat} score={round(score, 2)}')

category= lead_source score=0.03
category= industry score=0.0
category= employment_status score=0.02
category= location score=0.0


In [54]:
from sklearn.preprocessing import OneHotEncoder

In [80]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_cat = ohe.fit_transform(df_train[categorical].values)
X_train_cat

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(3000, 24))

In [81]:
ohe.categories_

[array(['NA', 'events', 'organic_search', 'paid_ads', 'referral',
        'social_media'], dtype=object),
 array(['NA', 'education', 'finance', 'healthcare', 'manufacturing',
        'retail', 'technology'], dtype=object),
 array(['NA', 'employed', 'self_employed', 'student', 'unemployed'],
       dtype=object),
 array(['NA', 'africa', 'asia', 'europe', 'north_america', 'south_america'],
       dtype=object)]

In [82]:
from sklearn.preprocessing import StandardScaler

In [83]:
X_train_num = df_train[numerical].values

#scaler = StandardScaler()

#X_train_num = scaler.fit_transform(X_train_num)
X_train_num

array([[6.5980e+04, 2.0000e+00, 3.0000e+00, 3.7000e-01],
       [7.1662e+04, 0.0000e+00, 0.0000e+00, 9.0000e-02],
       [4.0220e+04, 2.0000e+00, 5.0000e+00, 5.5000e-01],
       ...,
       [7.1532e+04, 3.0000e+00, 5.0000e+00, 5.8000e-01],
       [4.8836e+04, 0.0000e+00, 0.0000e+00, 3.0000e-02],
       [6.8969e+04, 0.0000e+00, 2.0000e+00, 3.4000e-01]], shape=(3000, 4))

In [84]:
X_train = np.column_stack([X_train_num, X_train_cat])
X_train

array([[6.5980e+04, 2.0000e+00, 3.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00],
       [7.1662e+04, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        1.0000e+00],
       [4.0220e+04, 2.0000e+00, 5.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00],
       ...,
       [7.1532e+04, 3.0000e+00, 5.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00],
       [4.8836e+04, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00],
       [6.8969e+04, 0.0000e+00, 2.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00]], shape=(3000, 28))

In [85]:
y_train = df_train.converted

In [86]:
from sklearn.linear_model import LogisticRegression

In [87]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

In [88]:
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [89]:
#X_val_num = StandardScaler().fit_transform(df_val[numerical].values)
X_val_num = df_val[numerical].values
X_val_cat = ohe.fit_transform(df_val[categorical].values)
X_val = np.column_stack([X_val_num, X_val_cat])
y_val = df_val.converted

In [90]:
from sklearn.metrics import accuracy_score

In [91]:
y_pred = model.predict_proba(X_val)[:, 1]

round(accuracy_score(y_val, y_pred >= 0.5),2)

0.65

In [92]:
y_pred = model.predict(X_val)
score_ref = accuracy_score(y_val, y_pred)
score_ref

0.645

In [93]:
numerical =['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
categorical =['lead_source', 'industry', 'employment_status', 'location']

y_train = df_train.converted
y_val = df_val.converted

X_train_cat = ohe.fit_transform(df_train[categorical].values)
X_val_cat = ohe.fit_transform(df_val[categorical].values)

for i in range(len(numerical)):
    num = numerical[:i]+numerical [i+1:]
    X_train_num = df_train[num].values
#    X_train_num = StandardScaler().fit_transform(df_train[num].values)
    X_train = np.column_stack([X_train_num, X_train_cat])

    X_val_num = df_val[num].values
#    X_val_num = StandardScaler().fit_transform(df_val[num].values)
    X_val = np.column_stack([X_val_num, X_val_cat])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    score = accuracy_score(y_val, y_pred)
    print(f'variable : {numerical[i]} dif ={score_ref - score}')

X_train_num = df_train[numerical].values
#X_train_num = StandardScaler().fit_transform(df_train[numerical].values)
X_val_num = df_val[numerical].values
#X_val_num = StandardScaler().fit_transform(df_val[numerical].values)

for i in range(len(categorical)):
    cat = categorical[:i]+categorical [i+1:]
    X_train_cat = ohe.fit_transform(df_train[cat].values)
    X_train = np.column_stack([X_train_num, X_train_cat])

    X_val_cat = ohe.fit_transform(df_val[cat].values)
    X_val = np.column_stack([X_val_num, X_val_cat])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    score = accuracy_score(y_val, y_pred)
    print(f'variable : {categorical[i]} dif ={score_ref - score}')    

variable : annual_income dif =-0.07899999999999996
variable : number_of_courses_viewed dif =0.0020000000000000018
variable : interaction_count dif =0.04400000000000004
variable : lead_score dif =0.0010000000000000009
variable : lead_source dif =0.0030000000000000027
variable : industry dif =0.0
variable : employment_status dif =0.0
variable : location dif =0.0


In [96]:
Cs = [0.000001, 0.00001, 0.0001, 0.001]

In [97]:
X_train_cat = ohe.fit_transform(df_train[categorical].values)
X_val_cat = ohe.fit_transform(df_val[categorical].values)
X_train_num = df_train[numerical].values
X_val_num = df_val[numerical].values
X_train = np.column_stack([X_train_num, X_train_cat])
X_val = np.column_stack([X_val_num, X_val_cat])

y_train = df_train.converted
y_val = df_val.converted

for C in Cs:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    score = round(accuracy_score(y_val, y_pred),3)
    print(f'C: {C} score={score}')


C: 1e-06 score=0.598
C: 1e-05 score=0.598
C: 0.0001 score=0.613
C: 0.001 score=0.645
